# EV Service Intelligence Platform — Inference Pipeline

## Objective

Build a reusable inference pipeline using the finalized trained models.

Given a new EV service visit, the pipeline will generate:

1. Predicted Repair Cost
2. Predicted Turnaround Time
3. Delay Risk probability
4. Delay Risk classification

## Production Flow

Raw service input
        ↓
Saved preprocessing + ML models
        ↓
Predictions
        ↓
Business-ready service intelligence output

The pipeline must use the same preprocessing and feature structure
used during model training.

In [1]:
import pandas as pd
import numpy as np
import joblib

In [6]:
df = pd.read_csv(
    "ml_ready.csv"
)

print("Dataset Shape:", df.shape)

Dataset Shape: (6583, 26)


In [7]:
repair_model = joblib.load(
    "repair_cost_model.pkl"
)

tat_model = joblib.load(
    "tat_model.pkl"
)

delay_model = joblib.load(
    "delay_model.pkl"
)

print("All production models loaded successfully.")

All production models loaded successfully.


In [8]:
feature_columns = [
    "Visit_Number",
    "Vehicle_Model",
    "Vehicle_Age_at_Service",
    "Battery_Age_at_Service",
    "Battery_Health_at_Service",
    "Battery_Replaced",
    "Issue_Family",
    "Exact_Issue",
    "Parts_Required",
    "Parts_Available",
    "Part_Ordered",
    "Expected_Part_ETA_Days",
    "Active_Jobs_On_Arrival",
    "Workshop_Utilization",
    "Day_Type",
    "Technician_Experience_Years",
    "Repair_Complexity",
    "Base_Labor_Hours",
    "Technician_Efficiency",
    "Effective_Labor_Hours",
    "Warranty_Status",
    "Warranty_Covered",
    "Expected_TAT_Days"
]

In [9]:
target_columns = [
    "Repair_Cost",
    "Turnaround_Time_Days",
    "Is_Delayed"
]

feature_columns = [
    col for col in df.columns
    if col not in target_columns
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 23
['Visit_Number', 'Vehicle_Model', 'Vehicle_Age_at_Service', 'Battery_Age_at_Service', 'Battery_Health_at_Service', 'Battery_Replaced', 'Issue_Family', 'Exact_Issue', 'Parts_Required', 'Parts_Available', 'Part_Ordered', 'Expected_Part_ETA_Days', 'Active_Jobs_On_Arrival', 'Workshop_Utilization', 'Day_Type', 'Technician_Experience_Years', 'Repair_Complexity', 'Base_Labor_Hours', 'Technician_Efficiency', 'Effective_Labor_Hours', 'Warranty_Status', 'Warranty_Covered', 'Expected_TAT_Days']


In [10]:
assert len(feature_columns) == 23

print("Feature contract validated.")

Feature contract validated.


In [11]:
new_service = pd.DataFrame([{
    "Visit_Number": 3,
    "Vehicle_Model": "Nexon EV",
    "Vehicle_Age_at_Service": 4,
    "Battery_Age_at_Service": 3,
    "Battery_Health_at_Service": 82,
    "Battery_Replaced": False,
    "Issue_Family": "Brake",
    "Exact_Issue": "Brake Pad Wear",
    "Parts_Required": True,
    "Parts_Available": True,
    "Part_Ordered": False,
    "Expected_Part_ETA_Days": 0,
    "Active_Jobs_On_Arrival": 8,
    "Workshop_Utilization": 0.55,
    "Day_Type": "Weekday",
    "Technician_Experience_Years": 5,
    "Repair_Complexity": 2,
    "Base_Labor_Hours": 2.5,
    "Technician_Efficiency": 0.90,
    "Effective_Labor_Hours": 2.25,
    "Warranty_Status": "Active",
    "Warranty_Covered": True,
    "Expected_TAT_Days": 1.2
}])

In [12]:
print("Input shape:", new_service.shape)

print("\nMissing columns:")
print(
    set(feature_columns) -
    set(new_service.columns)
)

print("\nExtra columns:")
print(
    set(new_service.columns) -
    set(feature_columns)
)

Input shape: (1, 23)

Missing columns:
set()

Extra columns:
set()


In [13]:
assert list(new_service.columns) == feature_columns

print("Input schema validated successfully.")

Input schema validated successfully.


In [14]:
predicted_repair_cost = repair_model.predict(
    new_service
)[0]

predicted_tat = tat_model.predict(
    new_service
)[0]

predicted_delay_probability = delay_model.predict_proba(
    new_service
)[0, 1]

predicted_delay = int(
    predicted_delay_probability >= 0.50
)

In [15]:
print("===== EV SERVICE INTELLIGENCE =====")

print(
    f"Predicted Repair Cost : "
    f"₹{predicted_repair_cost:.2f}"
)

print(
    f"Predicted TAT         : "
    f"{predicted_tat:.2f} days"
)

print(
    f"Delay Probability     : "
    f"{predicted_delay_probability:.1%}"
)

print(
    f"Delay Risk            : "
    f"{'HIGH' if predicted_delay == 1 else 'LOW'}"
)

===== EV SERVICE INTELLIGENCE =====
Predicted Repair Cost : ₹2016.71
Predicted TAT         : 0.47 days
Delay Probability     : 3.0%
Delay Risk            : LOW


In [16]:
prediction_output = {
    "Predicted_Repair_Cost": round(
        predicted_repair_cost, 2
    ),
    "Predicted_TAT_Days": round(
        predicted_tat, 2
    ),
    "Delay_Probability": round(
        predicted_delay_probability, 3
    ),
    "Delay_Risk": (
        "HIGH"
        if predicted_delay == 1
        else "LOW"
    )
}

prediction_output

{'Predicted_Repair_Cost': np.float64(2016.71),
 'Predicted_TAT_Days': np.float64(0.47),
 'Delay_Probability': np.float64(0.03),
 'Delay_Risk': 'LOW'}

In [17]:
prediction_output = {
    "Predicted_Repair_Cost": float(
        round(predicted_repair_cost, 2)
    ),
    "Predicted_TAT_Days": float(
        round(predicted_tat, 2)
    ),
    "Delay_Probability": float(
        round(predicted_delay_probability, 3)
    ),
    "Delay_Risk": (
        "HIGH"
        if predicted_delay == 1
        else "LOW"
    )
}

print(prediction_output)

{'Predicted_Repair_Cost': 2016.71, 'Predicted_TAT_Days': 0.47, 'Delay_Probability': 0.03, 'Delay_Risk': 'LOW'}


In [18]:
def predict_service_outcome(service_data):

    input_df = pd.DataFrame(
        [service_data]
    )

    # Validate schema
    assert list(input_df.columns) == feature_columns, \
        "Input features do not match the production feature schema."

    # Predictions
    repair_cost = repair_model.predict(
        input_df
    )[0]

    tat_days = tat_model.predict(
        input_df
    )[0]

    delay_probability = delay_model.predict_proba(
        input_df
    )[0, 1]

    delay_risk = (
        "HIGH"
        if delay_probability >= 0.50
        else "LOW"
    )

    return {
        "Predicted_Repair_Cost": float(
            round(repair_cost, 2)
        ),
        "Predicted_TAT_Days": float(
            round(tat_days, 2)
        ),
        "Delay_Probability": float(
            round(delay_probability, 3)
        ),
        "Delay_Risk": delay_risk
    }

In [19]:
result = predict_service_outcome(
    new_service.iloc[0].to_dict()
)

print(result)

{'Predicted_Repair_Cost': 2016.71, 'Predicted_TAT_Days': 0.47, 'Delay_Probability': 0.03, 'Delay_Risk': 'LOW'}


In [20]:
expected_output_keys = [
    "Predicted_Repair_Cost",
    "Predicted_TAT_Days",
    "Delay_Probability",
    "Delay_Risk"
]

assert list(result.keys()) == expected_output_keys

print("Output schema validated successfully.")

Output schema validated successfully.


In [21]:
assert result["Predicted_Repair_Cost"] >= 0
assert result["Predicted_TAT_Days"] >= 0

assert 0 <= result["Delay_Probability"] <= 1

assert result["Delay_Risk"] in [
    "HIGH",
    "LOW"
]

print("Prediction ranges validated successfully.")

Prediction ranges validated successfully.


In [22]:
test_inputs = df[
    feature_columns
].iloc[
    [0, 100, 500, 1000]
]

for i, row in test_inputs.iterrows():

    result = predict_service_outcome(
        row.to_dict()
    )

    print(f"\nService Row: {i}")
    print(result)


Service Row: 0
{'Predicted_Repair_Cost': 3237.27, 'Predicted_TAT_Days': 2.19, 'Delay_Probability': 0.416, 'Delay_Risk': 'LOW'}

Service Row: 100
{'Predicted_Repair_Cost': 493.16, 'Predicted_TAT_Days': 0.31, 'Delay_Probability': 0.103, 'Delay_Risk': 'LOW'}

Service Row: 500
{'Predicted_Repair_Cost': 3594.07, 'Predicted_TAT_Days': 0.7, 'Delay_Probability': 0.033, 'Delay_Risk': 'LOW'}

Service Row: 1000
{'Predicted_Repair_Cost': 3586.51, 'Predicted_TAT_Days': 0.92, 'Delay_Probability': 0.006, 'Delay_Risk': 'LOW'}


## Inference Pipeline — Final Status

The production inference pipeline successfully:

- Loads all three finalized ML models.
- Validates the 23-feature input schema.
- Generates Repair Cost predictions.
- Generates Turnaround Time predictions.
- Generates Delay Risk probability.
- Converts Delay Risk probability into a business classification.
- Returns JSON-compatible Python values.
- Validates prediction ranges.
- Successfully processes multiple service visits.

### Production Output

For a new EV service visit, the pipeline returns:

1. Predicted Repair Cost
2. Predicted TAT
3. Delay Probability
4. Delay Risk

The inference pipeline is ready to be exposed through an API.